In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [2]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [3]:
train = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/train.csv')
test = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv').drop(columns=['id'])

In [4]:
train['humidity'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in train['humidity'].to_numpy()])
train['pressure'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in train['pressure'].to_numpy()])
train['wind_speed'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in train['wind_speed'].to_numpy()])

test['humidity'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in test['humidity'].to_numpy()])
test['pressure'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in test['pressure'].to_numpy()])
test['wind_speed'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in test['wind_speed'].to_numpy()])

column = list(train.columns)
column.remove('efficiency')
column.remove('id')
column.remove('string_id')
column.remove('error_code')
column.remove('installation_type')

for i_ in range(len(column)):
    for j in column[i_:]:
        i = column[i_]
        if i != j:
            print(f'{i} x {j}')
            train[f'{i} x {j}'] = (train[i].astype('float').to_numpy() * train[j].astype('float').to_numpy())
            test[f'{i} x {j}'] = (test[i].astype('float').to_numpy() * test[j].astype('float').to_numpy())

train

temperature x irradiance
temperature x humidity
temperature x panel_age
temperature x maintenance_count
temperature x soiling_ratio
temperature x voltage
temperature x current
temperature x module_temperature
temperature x cloud_coverage
temperature x wind_speed
temperature x pressure
irradiance x humidity
irradiance x panel_age
irradiance x maintenance_count
irradiance x soiling_ratio
irradiance x voltage
irradiance x current
irradiance x module_temperature
irradiance x cloud_coverage
irradiance x wind_speed
irradiance x pressure
humidity x panel_age
humidity x maintenance_count
humidity x soiling_ratio
humidity x voltage
humidity x current
humidity x module_temperature
humidity x cloud_coverage
humidity x wind_speed
humidity x pressure
panel_age x maintenance_count
panel_age x soiling_ratio
panel_age x voltage
panel_age x current
panel_age x module_temperature
panel_age x cloud_coverage
panel_age x wind_speed
panel_age x pressure
maintenance_count x soiling_ratio
maintenance_count x 

,id,temperature,irradiance,humidity,panel_age,maintenance_count,soiling_ratio,voltage,current,module_temperature,...,current x module_temperature,current x cloud_coverage,current x wind_speed,current x pressure,module_temperature x cloud_coverage,module_temperature x wind_speed,module_temperature x pressure,cloud_coverage x wind_speed,cloud_coverage x pressure,wind_speed x pressure
0,0,7.817315,576.179270,41.243087,32.135501,4.0,0.803199,37.403527,1.963787,13.691147,...,26.886498,122.724995,25.185396,2000.836842,855.615169,175.587760,13949.451427,801.480623,63673.088659,13066.873306
1,1,24.785727,240.003973,1.359648,19.977460,8.0,0.479456,21.843315,0.241473,27.545096,...,6.651405,10.588903,2.900588,247.660768,1207.886573,330.872897,28250.907613,526.742989,44974.876015,12319.838511
2,2,46.652695,687.612799,91.265368,1.496401,4.0,0.822398,48.222882,4.191800,43.363708,...,181.772002,NaN,7.605601,4237.585828,NaN,78.679101,43837.354846,NaN,NaN,1834.217816
3,3,53.339567,735.141179,96.190955,18.491582,3.0,0.837529,46.295748,0.960567,57.720436,...,55.444346,64.705208,8.391762,981.552185,3888.133592,504.260676,58981.435103,588.487269,68833.096242,8927.117040
4,4,5.575374,12.241203,27.495073,30.722697,6.0,0.551833,0.000000,0.898062,6.786263,...,6.094485,3.261762,0.469403,905.745862,24.647706,3.547070,6844.325709,1.898388,3663.075265,527.155902
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,19995,16.868428,NaN,93.530318,14.393967,3.0,0.738911,12.147711,3.005355,26.206810,...,78.760768,5.208320,37.849809,3060.576822,45.416750,330.051767,26688.345972,21.825781,1764.856503,12825.532558
19996,19996,53.415061,296.970303,93.985714,25.997012,2.0,0.513061,0.000000,0.532119,65.000000,...,34.587718,34.352876,0.519875,540.675788,4196.313359,63.504410,66045.271634,63.073232,65596.841568,992.702020
19997,19997,2.442727,660.328019,37.968918,32.818396,9.0,0.548602,13.047950,4.075498,11.584869,...,47.214110,235.279048,19.362436,4114.967085,668.796015,55.038984,11697.061875,274.272242,58289.218796,4796.947519
19998,19998,NaN,632.760700,43.014702,19.063517,4.0,NaN,0.000000,1.068906,21.149351,...,22.606671,83.506890,12.083084,1076.039878,1652.265287,239.075610,21290.498676,883.122559,78645.076778,11379.600979


In [5]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [6]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]]=normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]].fillna(train[i[0]].mean(), inplace=True)
        train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]]=normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        test[i[0]].fillna(test[i[0]].mean(), inplace=True)
        test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [7]:
train_x = train.drop(columns=['id', 'efficiency'])
train_y = train['efficiency']

In [8]:
LGBM_R_parm = {'colsample_bytree': 0.7127886176981258,
               'drop_rate': 0.8124406739963222,
               'learning_rate': 0.07912182737635276,
               'max_bin': 6651,
               'max_depth': 9050,
               'max_drop': 8493,
               'min_child_samples': 6754,
               'min_data_in_leaf': 2929,
               'n_estimators': 25802,
               'num_leaves': 61,
               'objective': 'regression_l1',
               'reg_alpha': 0.881842406619579,
               'reg_lambda': 0.456201832946564,
               'skip_drop': 0.07580175182420101,
               'verbosity': -1}

XGB_R_parm = {'colsample_bytree': 0.6146156360042275,
              'gamma': 1,
              'learning_rate': 0.7129688111132676,
              'max_depth': 5,
              'min_child_weight': 1,
              'n_estimators': 3145,
              'objective': 'binary:logistic',
              'reg_alpha': 85,
              'reg_lambda': 0.27992509484698325,
              'subsample': 0.7284972930896713}

X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size=0.1, random_state=100)

In [9]:
LGBM_R = LGBMRegressor(**LGBM_R_parm)

LGBM_R.fit(X_train, 
           y_train,
           eval_set = [(X_test, y_test)],
           eval_metric ='rmse')

LGBMRegressor(colsample_bytree=0.7127886176981258, drop_rate=0.8124406739963222,
              learning_rate=0.07912182737635276, max_bin=6651, max_depth=9050,
              max_drop=8493, min_child_samples=6754, min_data_in_leaf=2929,
              n_estimators=25802, num_leaves=61, objective='regression_l1',
              reg_alpha=0.881842406619579, reg_lambda=0.456201832946564,
              skip_drop=0.07580175182420101, verbosity=-1)

In [10]:
XGB_R = XGBRegressor(**XGB_R_parm)
XGB_R.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6146156360042275, device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, gamma=1, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.7129688111132676, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=1, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=3145, n_jobs=None,
             num_parallel_tree=None, objective='binary:logistic', ...)

In [11]:
catboost = CatBoostRegressor(learning_rate=0.009,
                             depth=5,
                             l2_leaf_reg=5.5,
                             min_child_samples=102,
                             grow_policy='Depthwise',
                             iterations=1000,
                             eval_metric='RMSE',
                             od_type='Iter',
                             od_wait=50,
                             random_state=42,
                             logging_level='Silent')

cat_features = list(train_x.select_dtypes(include=['object', 'category']).columns)
    
train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool = Pool(X_test, y_test, cat_features=cat_features)
        
catboost.fit(train_pool, eval_set=(val_pool), verbose=100, early_stopping_rounds=100)

In [12]:
cat_pre_train = catboost.predict(X_train)
LGBM_pre_train = LGBM_R.predict(X_train)
XGB_pre_train = XGB_R.predict(X_train)

cat_pre_test_y = catboost.predict(X_test)
LGBM_pre_test_y = LGBM_R.predict(X_test)
XGB_pre_test_y = XGB_R.predict(X_test)

cat_pre_test = catboost.predict(test)
LGBM_pre_test = LGBM_R.predict(test)
XGB_pre_test = XGB_R.predict(test)

In [13]:
new_train_data = pandas.DataFrame({'LGBM' : LGBM_pre_train * 0.25,
                                   'xgb' : XGB_pre_train * 0.25,
                                   'cat' : cat_pre_train * 0.5})

new_test_x_data = pandas.DataFrame({'LGBM' : LGBM_pre_test_y * 0.25,
                                  'xgb' : XGB_pre_test_y * 0.25,
                                  'cat' : cat_pre_test_y * 0.5})

new_test_data = pandas.DataFrame({'LGBM' : LGBM_pre_test * 0.25,
                                  'xgb' : XGB_pre_test * 0.25,
                                  'cat' : cat_pre_test * 0.5})

In [14]:
model = RidgeCV()
model.fit(new_train_data, y_train)

print(f'Concordance index (lifelines): {100*(1-numpy.sqrt(mean_squared_error(y_test, model.predict(new_test_x_data))))}')

Concordance index (lifelines): 90.2468005024483


In [15]:
id = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv')['id']
test_predictions = model.predict(new_test_data)
test_predictions

array([0.37346718, 0.51963349, 0.51045461, ..., 0.61544439, 0.42703488,
       0.55014051])

In [16]:
submission = pandas.DataFrame({
    'id': id.values,
    'efficiency': test_predictions,
})
submission

,id,efficiency
0,0,0.373467
1,1,0.519633
2,2,0.510455
3,3,0.478493
4,4,0.483398
...,...,...
11995,11995,0.547665
11996,11996,0.467240
11997,11997,0.615444
11998,11998,0.427035


In [17]:
submission.to_csv('submission.csv', index = False)